In [0]:
# =====================================================================================
# Notebook: 02_silver/03_odds_h2h_silver.py
# Project : nba-datalakehouse
# Purpose : Convert Odds API Bronze -> Silver for head-to-head (h2h) odds.
#
# Bronze contract (you confirmed):
# - dt: string
# - meta: string (json)
# - data: string (json)  <-- each row is one event odds object
#
# Silver outputs:
# 1) workspace.nba_silver.fact_odds_h2h_history
#    - append-only style snapshot by source_dt (bronze dt)
#    - grain: (event_id, bookmaker_key, market_key, outcome_name, source_dt)
#
# 2) workspace.nba_silver.fact_odds_h2h_latest
#    - latest record per (event_id, bookmaker_key, market_key, outcome_name)
#    - easy for analytics ("what are the latest odds?")
# =====================================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_ODDS = "workspace.nba_bronze.the_odds_api_odds"

SILVER_ODDS_H2H_HISTORY = "workspace.nba_silver.fact_odds_h2h_history"
SILVER_ODDS_H2H_LATEST = "workspace.nba_silver.fact_odds_h2h_latest"

# -----------------------------
# 0) Helper: infer + parse JSON string column into struct
# -----------------------------
def infer_json_schema(df, json_col: str):
    sample = (
        df.select(F.col(json_col))
          .where(F.col(json_col).isNotNull())
          .limit(1)
          .collect()
    )
    if not sample:
        raise RuntimeError(f"Cannot infer schema: no non-null values found in {json_col}")
    return spark.range(1).select(F.schema_of_json(F.lit(sample[0][0])).alias("schema")).collect()[0]["schema"]

def parse_json_col(df, json_col: str, struct_col: str):
    schema_str = infer_json_schema(df, json_col)
    return df.withColumn(struct_col, F.from_json(F.col(json_col), schema_str))

# -----------------------------
# 1) Load + parse Bronze odds
# -----------------------------
odds_bronze = spark.table(BRONZE_ODDS)

odds_parsed = (
    parse_json_col(odds_bronze, "data", "data_s")
    .select(
        F.col("dt").cast("date").alias("source_dt"),
        F.col("_loaded_at").alias("ingested_at"),
        F.col("data_s.id").alias("event_id"),
        F.col("data_s.sport_key").alias("sport_key"),
        F.col("data_s.sport_title").alias("sport_title"),
        F.to_timestamp(F.col("data_s.commence_time")).alias("commence_time_utc"),
        F.col("data_s.home_team").alias("home_team_name"),
        F.col("data_s.away_team").alias("away_team_name"),
        F.col("data_s.bookmakers").alias("bookmakers"),
    )
    .filter(F.col("event_id").isNotNull())
)

# -----------------------------
# 2) Flatten bookmakers -> markets -> outcomes
# -----------------------------
bm = odds_parsed.select(
    "source_dt","ingested_at","event_id","sport_key","sport_title","commence_time_utc",
    "home_team_name","away_team_name",
    F.explode_outer("bookmakers").alias("bm")
)

mk = bm.select(
    "source_dt","ingested_at","event_id","sport_key","sport_title","commence_time_utc",
    "home_team_name","away_team_name",
    F.col("bm.key").alias("bookmaker_key"),
    F.col("bm.title").alias("bookmaker_title"),
    F.to_timestamp(F.col("bm.last_update")).alias("bookmaker_last_update_utc"),
    F.explode_outer(F.col("bm.markets")).alias("mk")
)

oc = mk.select(
    "source_dt","ingested_at","event_id","sport_key","sport_title","commence_time_utc",
    "home_team_name","away_team_name",
    "bookmaker_key","bookmaker_title","bookmaker_last_update_utc",
    F.col("mk.key").alias("market_key"),
    F.to_timestamp(F.col("mk.last_update")).alias("market_last_update_utc"),
    F.explode_outer(F.col("mk.outcomes")).alias("oc")
)

# Keep only head-to-head (moneyline) market for now
h2h = (
    oc.filter(F.col("market_key") == F.lit("h2h"))
      .select(
          "source_dt","ingested_at",
          "event_id","sport_key","sport_title","commence_time_utc",
          "home_team_name","away_team_name",
          "bookmaker_key","bookmaker_title",
          "bookmaker_last_update_utc","market_key","market_last_update_utc",
          F.col("oc.name").alias("outcome_name"),
          F.col("oc.price").cast("double").alias("price_decimal"),
      )
      .filter(F.col("outcome_name").isNotNull() & F.col("price_decimal").isNotNull())
      .withColumn("source_system", F.lit("the_odds_api"))
)

# -----------------------------
# 3) Write HISTORY (v1 rebuild semantics)
# -----------------------------
# Since your Bronze loads are partitioned by dt, the simplest robust v1 is to overwrite the entire table.
# Later, we can incrementalize by overwriting only source_dt partitions.

spark.sql(f"DROP TABLE IF EXISTS {SILVER_ODDS_H2H_HISTORY}")

(h2h.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("source_dt")
 .saveAsTable(SILVER_ODDS_H2H_HISTORY))

print("Wrote:", SILVER_ODDS_H2H_HISTORY, "rows=", h2h.count())

# -----------------------------
# 4) Build LATEST snapshot
# -----------------------------
# Latest per (event_id, bookmaker_key, market_key, outcome_name)
# We order by:
#   market_last_update_utc (best indicator)
#   then ingested_at (fallback)

w = (
    Window.partitionBy("event_id","bookmaker_key","market_key","outcome_name")
          .orderBy(F.col("market_last_update_utc").desc_nulls_last(), F.col("ingested_at").desc())
)

h2h_latest = (
    h2h.withColumn("_rn", F.row_number().over(w))
       .filter(F.col("_rn") == 1)
       .drop("_rn")
)

spark.sql(f"DROP TABLE IF EXISTS {SILVER_ODDS_H2H_LATEST}")

(h2h_latest.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(SILVER_ODDS_H2H_LATEST))

print("Wrote:", SILVER_ODDS_H2H_LATEST, "rows=", h2h_latest.count())
display(h2h_latest.orderBy(F.col("commence_time_utc").desc(), F.col("bookmaker_key")).limit(30))

# -----------------------------
# 5) Sanity checks
# -----------------------------
print("\n=== dt coverage (history) ===")
spark.table(SILVER_ODDS_H2H_HISTORY).groupBy("source_dt").count().orderBy(F.col("source_dt").desc()).show(30, truncate=False)

print("\n=== Sample latest for a single event_id ===")
sample_event = h2h_latest.select("event_id").limit(1).collect()[0]["event_id"]
spark.table(SILVER_ODDS_H2H_LATEST).filter(F.col("event_id") == sample_event) \
    .orderBy(F.col("bookmaker_key")) \
    .select("event_id","commence_time_utc","home_team_name","away_team_name","bookmaker_key","outcome_name","price_decimal","market_last_update_utc") \
    .show(200, truncate=False)
